In [3]:
import os
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter

def rechunk_jsonl_files(input_folder, output_folder, chunk_size=500, chunk_overlap=50):
    # 출력 폴더 생성 (04_chunks/first)
    os.makedirs(output_folder, exist_ok=True)
    
    if not os.path.exists(input_folder):
        print(f"입력 경로를 찾을 수 없습니다: {input_folder}")
        return
        
    # .jsonl 파일만 골라내기
    files = [f for f in os.listdir(input_folder) if f.endswith('.jsonl')]
    
    if not files:
        print("api 폴더 내에 처리할 .jsonl 파일이 없습니다.")
        return

    # 텍스트 스플리터 설정
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    print(f"🔄 JSONL 재청킹 시작 (Size: {chunk_size}, Overlap: {chunk_overlap})")
    print("-" * 60)

    for file_name in files:
        input_path = os.path.join(input_folder, file_name)
        output_path = os.path.join(output_folder, file_name)
        
        new_chunks_count = 0
        
        try:
            # JSONL은 한 줄씩 읽고 써야 하므로 open 형식을 맞춰줍니다.
            with open(input_path, 'r', encoding='utf-8') as infile, \
                 open(output_path, 'w', encoding='utf-8') as outfile:
                
                for line in infile:
                    line = line.strip()
                    if not line:
                        continue  # 빈 줄은 건너뜀
                        
                    item = json.loads(line)
                    
                    if isinstance(item, dict) and 'page_content' in item:
                        base_content = item['page_content']
                        base_metadata = item.get('metadata', {})
                        
                        # 텍스트 분할
                        split_texts = text_splitter.split_text(base_content)
                        
                        # 쪼개진 청크들을 한 줄씩 바로 저장
                        for text in split_texts:
                            chunk_data = {
                                "page_content": text,
                                "metadata": base_metadata.copy()
                            }
                            # 한 줄씩 json 문자열로 변환 후 줄바꿈 기입
                            outfile.write(json.dumps(chunk_data, ensure_ascii=False) + "\n")
                            new_chunks_count += 1
                            
            print(f"✅ 완료: {file_name} -> 새 청크 {new_chunks_count}개 생성 완료")
            
        except Exception as e:
            print(f"❌ 오류 발생 ({file_name}): {e}")

    print("-" * 60)
    print(f"🎉 모든 api 데이터가 {output_folder} 폴더에 성공적으로 재청킹되었습니다.")

# --- 실행부 ---
# 프로젝트 구조에 맞춰 상대 경로 설정
input_dir = "../data/04_chunks/api"
output_dir = "../data/04_chunks/first"

rechunk_jsonl_files(input_dir, output_dir, chunk_size=500, chunk_overlap=50)

🔄 JSONL 재청킹 시작 (Size: 500, Overlap: 50)
------------------------------------------------------------
✅ 완료: kb_chunks_eflaw.jsonl -> 새 청크 2888개 생성 완료
------------------------------------------------------------
🎉 모든 api 데이터가 ../data/04_chunks/first 폴더에 성공적으로 재청킹되었습니다.
